# 04 - Comparação de Modelos

Este notebook dá sequência ao `03_modelo_baseline.ipynb`, testando algoritmos mais robustos que a Regressão Logística (baseline) nos dois cenários de modelagem já estabelecidos (`referencia` e `comportamental`), reutilizando a arquitetura modular do projeto (`src/model_training.py`, `src/model_evaluation.py` e `src/model_comparison.py`).

**Pergunta central:** algum algoritmo do cenário `comportamental` (sem `nr_altura`, `nr_peso` e `nr_imc`) consegue atingir a meta de 75% de acurácia exigida pelo desafio? O baseline (Regressão Logística) ficou em ~62,2% de accuracy no conjunto de teste — abaixo da meta — usando apenas hábitos de vida.

**Regra de seleção do melhor modelo:** a escolha do melhor modelo por cenário é feita pelas métricas de **validação cruzada** (`accuracy_cv_media`/`f1_macro_cv_media`), calculadas por `executar_comparacao()` sobre `X_train`/`y_train`. As métricas de teste também são calculadas e reportadas, mas apenas como informação complementar — nunca como critério de escolha, para não arriscar overfitting via comparação repetida sobre o mesmo conjunto de teste.

## 1. Importações

Bibliotecas padrão, de terceiros e código do projeto (`src/`), em um único bloco, agrupadas nessa ordem. Adicionamos `sys.path.append("..")` para que `src` seja importável a partir da pasta `notebooks/`, na mesma convenção usada em `01_eda.ipynb`, `02_preprocessamento_ml.ipynb` e `03_modelo_baseline.ipynb`.

Além dos quatro algoritmos já usados em versões anteriores do projeto (Regressão Logística, Árvore de Decisão, Random Forest e Gradient Boosting), incluímos aqui o `XGBClassifier` (biblioteca `xgboost`) como quinto candidato — um algoritmo de gradient boosting mais moderno e geralmente competitivo em dados tabulares.

In [1]:
import sys

import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

sys.path.append("..")
from src.model_comparison import executar_comparacao
from src.model_training import (
    META_ACCURACY,
    ORDEM_NIVEIS_OBESIDADE,
    RANDOM_STATE,
    TARGET,
    carregar_dados_silver,
    criar_cenarios_modelagem,
    separar_features_target,
)

## 2. Carregamento dos dados

Carregamos a camada Silver com `carregar_dados_silver()` — a mesma base já tratada e utilizada nos notebooks `02_preprocessamento_ml.ipynb` e `03_modelo_baseline.ipynb`.

In [2]:
dados_silver = carregar_dados_silver()
dados_silver.head()

,ds_genero,nr_idade,nr_altura,nr_peso,fl_historico_familiar_sobrepeso,fl_consumo_calorico_frequente,cd_consumo_de_vegetais,cd_numero_refeicoes_principais,ds_lanches_entre_refeicoes,fl_fumante,...,fl_monitora_calorias,cd_frequencia_atividade_fisica,cd_tempo_uso_eletronicos,ds_consumo_alcool,ds_meio_transporte,ds_nivel_obesidade,nr_imc,fl_transporte_ativo,ds_faixa_etaria,ds_consumo_alcool_agrupado
0,Female,21.0,1.62,64.0,1,0,2,3,Sometimes,0,...,0,0,1,no,Public_Transportation,Normal_Weight,24.386526,0,adulto_jovem,no
1,Female,21.0,1.52,56.0,1,0,3,3,Sometimes,1,...,1,3,0,Sometimes,Public_Transportation,Normal_Weight,24.238227,0,adulto_jovem,Sometimes
2,Male,23.0,1.80,77.0,1,0,2,3,Sometimes,0,...,0,2,1,Frequently,Public_Transportation,Normal_Weight,23.765432,0,adulto_jovem,consumo_frequente_ou_mais
3,Male,27.0,1.80,87.0,0,0,3,3,Sometimes,0,...,0,2,0,Frequently,Walking,Overweight_Level_I,26.851852,1,adulto_jovem,consumo_frequente_ou_mais
4,Male,22.0,1.78,89.8,0,0,2,1,Sometimes,0,...,0,0,0,Sometimes,Public_Transportation,Overweight_Level_II,28.342381,0,adulto_jovem,Sometimes


## 3. Separação de atributos e variável alvo

Separamos `X` (atributos preditores) de `y` (`TARGET`, importado de `src/model_training.py`) com `separar_features_target()`, antes de criar os cenários de modelagem.

In [3]:
X, y = separar_features_target(dados_silver, target=TARGET)
X.shape, y.shape

((2087, 20), (2087,))

## 4. Criação dos cenários

Reconstruímos os dois cenários de modelagem com `criar_cenarios_modelagem()` — a mesma função usada nos notebooks anteriores, garantindo que `referencia` (com métricas antropométricas) e `comportamental` (sem elas) sejam definidos de forma idêntica em todo o projeto.

In [4]:
cenarios = criar_cenarios_modelagem(X)

for nome_cenario, X_cenario in cenarios.items():
    print(f"Cenário '{nome_cenario}': {X_cenario.shape[1]} colunas -> {list(X_cenario.columns)}")

Cenário 'referencia': 20 colunas -> ['ds_genero', 'nr_idade', 'nr_altura', 'nr_peso', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'ds_lanches_entre_refeicoes', 'fl_fumante', 'cd_consumo_agua', 'fl_monitora_calorias', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos', 'ds_consumo_alcool', 'ds_meio_transporte', 'nr_imc', 'fl_transporte_ativo', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']
Cenário 'comportamental': 17 colunas -> ['ds_genero', 'nr_idade', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'ds_lanches_entre_refeicoes', 'fl_fumante', 'cd_consumo_agua', 'fl_monitora_calorias', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos', 'ds_consumo_alcool', 'ds_meio_transporte', 'fl_transporte_ativo', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']


## 5. Definição dos modelos

Testamos cinco algoritmos nos dois cenários: **Regressão Logística** (o mesmo baseline do `03_modelo_baseline.ipynb`, incluída aqui para servir de referência direta de comparação), **Árvore de Decisão**, **Random Forest**, **Gradient Boosting** e **XGBoost**. Todos usam `random_state=RANDOM_STATE` (importado de `src/model_training.py`) — nenhuma semente é hardcoded neste notebook.

**Particularidade do XGBoost:** diferente dos demais algoritmos (que aceitam `y` como texto, ex.: `"Normal_Weight"`), o `XGBClassifier` exige que o alvo seja codificado como inteiros (`0`, `1`, `2`, ...) — ele levanta erro se receber classes em formato string. Para que o XGBoost possa ser comparado aos demais através da mesma `executar_comparacao()` (que recebe um único `y` em texto, compartilhado por todos os modelos), criamos um pequeno wrapper (`XGBoostComEncodingDeAlvo`) que aplica um `LabelEncoder` internamente antes de treinar e reverte a codificação em `predict()`. Assim, por fora, o wrapper se comporta como qualquer outro estimador do Scikit-Learn — recebe e devolve `y` em texto —, mantendo `avaliar_modelo()` e a ordem clínica de `ORDEM_NIVEIS_OBESIDADE` funcionando sem nenhuma alteração em `src/model_evaluation.py` ou `src/model_comparison.py`.

In [5]:
class XGBoostComEncodingDeAlvo(BaseEstimator, ClassifierMixin):
    """Wrapper do XGBClassifier que aceita o alvo como texto.

    O XGBClassifier exige y codificado como inteiros; este wrapper aplica
    um LabelEncoder internamente em fit() e reverte a codificação em
    predict(), para que o restante do projeto (avaliar_modelo,
    ORDEM_NIVEIS_OBESIDADE) continue operando sobre y em texto.
    """

    def __init__(self, random_state=RANDOM_STATE):
        self.random_state = random_state

    def fit(self, X, y):
        self.codificador_alvo_ = LabelEncoder()
        y_codificado = self.codificador_alvo_.fit_transform(y)

        self.modelo_ = XGBClassifier(random_state=self.random_state)
        self.modelo_.fit(X, y_codificado)

        self.classes_ = self.codificador_alvo_.classes_

        return self

    def predict(self, X):
        y_pred_codificado = self.modelo_.predict(X)
        return self.codificador_alvo_.inverse_transform(y_pred_codificado)

In [6]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBoostComEncodingDeAlvo(random_state=RANDOM_STATE),
}

list(modelos.keys())

['Logistic Regression',
 'Decision Tree',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## 6. Comparação

`executar_comparacao()` (`src/model_comparison.py`) treina e avalia todas as combinações de cenário x modelo de forma padronizada: para cada uma, roda validação cruzada estratificada sobre `X_train`/`y_train`, treina a pipeline final e a avalia em `X_test`/`y_test` (com `labels=ORDEM_NIVEIS_OBESIDADE`). Internamente, a função também garante que todos os cenários compartilhem exatamente a mesma divisão de treino/teste, para que a comparação entre eles seja justa.

A função retorna uma tupla: um DataFrame resumo (`resultado`) e um dicionário (`detalhes`), indexado por `(cenario, modelo)`, com a pipeline já treinada e o diagnóstico completo de teste (`classification_report`, `matriz_confusao`) de cada combinação — para uso nas próximas seções sem re-treinar nada.

In [7]:
pd.set_option("display.max_columns", None)

resultado, detalhes = executar_comparacao(
    cenarios=cenarios,
    y=y,
    modelos=modelos,
)
resultado

,cenario,modelo,accuracy_cv_media,accuracy_cv_desvio,f1_macro_cv_media,f1_macro_cv_desvio,accuracy_teste,f1_macro_teste,fit_time
0,referencia,Logistic Regression,0.899338,0.022703,0.894821,0.023886,0.918660,0.915183,0.420711
1,referencia,Decision Tree,0.969441,0.003505,0.968707,0.003259,0.980861,0.980391,0.023661
2,referencia,Random Forest,0.980232,0.006441,0.979263,0.006703,0.976077,0.975247,0.647997
3,referencia,Gradient Boosting,0.973635,0.002269,0.973176,0.002324,0.980861,0.980095,2.579855
4,referencia,XGBoost,0.979026,0.004257,0.978450,0.004651,0.988038,0.987502,2.069844
5,comportamental,Logistic Regression,0.607540,0.032639,0.581662,0.035337,0.622010,0.588067,0.255145
6,comportamental,Decision Tree,0.726781,0.008394,0.717475,0.010577,0.717703,0.709633,0.021904
7,comportamental,Random Forest,0.796289,0.006171,0.789098,0.007303,0.794258,0.787097,0.285486
8,comportamental,Gradient Boosting,0.757933,0.020061,0.750872,0.023137,0.765550,0.757677,1.929354
9,comportamental,XGBoost,0.809466,0.009409,0.803079,0.009721,0.775120,0.769354,3.099699


## 7. Ranking

Ordenamos as combinações por `accuracy_cv_media` e `f1_macro_cv_media` — as métricas de validação cruzada, que são o critério de seleção do projeto (o DataFrame retornado por `executar_comparacao()` não tem mais as colunas antigas `accuracy`/`f1_macro`, que vinham de um único split de teste). Também marcamos explicitamente, com duas colunas booleanas, quais combinações atingem a meta de 75% de acurácia exigida pelo desafio (`META_ACCURACY`, importada de `src/model_training.py`): `bate_meta_cv` (pela validação cruzada) e `bate_meta_teste` (pelo teste, apenas para conferência).

In [8]:
ranking = resultado.sort_values(
    by=["accuracy_cv_media", "f1_macro_cv_media"],
    ascending=False,
).reset_index(drop=True)

ranking["bate_meta_cv"] = ranking["accuracy_cv_media"] >= META_ACCURACY
ranking["bate_meta_teste"] = ranking["accuracy_teste"] >= META_ACCURACY

ranking

,cenario,modelo,accuracy_cv_media,accuracy_cv_desvio,f1_macro_cv_media,f1_macro_cv_desvio,accuracy_teste,f1_macro_teste,fit_time,bate_meta_cv,bate_meta_teste
0,referencia,Random Forest,0.980232,0.006441,0.979263,0.006703,0.976077,0.975247,0.647997,True,True
1,referencia,XGBoost,0.979026,0.004257,0.978450,0.004651,0.988038,0.987502,2.069844,True,True
2,referencia,Gradient Boosting,0.973635,0.002269,0.973176,0.002324,0.980861,0.980095,2.579855,True,True
3,referencia,Decision Tree,0.969441,0.003505,0.968707,0.003259,0.980861,0.980391,0.023661,True,True
4,referencia,Logistic Regression,0.899338,0.022703,0.894821,0.023886,0.918660,0.915183,0.420711,True,True
5,comportamental,XGBoost,0.809466,0.009409,0.803079,0.009721,0.775120,0.769354,3.099699,True,True
6,comportamental,Random Forest,0.796289,0.006171,0.789098,0.007303,0.794258,0.787097,0.285486,True,True
7,comportamental,Gradient Boosting,0.757933,0.020061,0.750872,0.023137,0.765550,0.757677,1.929354,True,True
8,comportamental,Decision Tree,0.726781,0.008394,0.717475,0.010577,0.717703,0.709633,0.021904,False,False
9,comportamental,Logistic Regression,0.607540,0.032639,0.581662,0.035337,0.622010,0.588067,0.255145,False,False


In [9]:
print(f"Meta de accuracy do desafio: {META_ACCURACY:.0%}\n")

print("Combinações que ATINGEM a meta (pela validação cruzada):")
for _, linha in ranking[ranking["bate_meta_cv"]].iterrows():
    print(
        f"  - {linha['cenario']} / {linha['modelo']}: "
        f"accuracy_cv_media = {linha['accuracy_cv_media']:.4f}"
    )

print("\nCombinações que NÃO atingem a meta (pela validação cruzada):")
for _, linha in ranking[~ranking["bate_meta_cv"]].iterrows():
    print(
        f"  - {linha['cenario']} / {linha['modelo']}: "
        f"accuracy_cv_media = {linha['accuracy_cv_media']:.4f}"
    )

Meta de accuracy do desafio: 75%

Combinações que ATINGEM a meta (pela validação cruzada):
  - referencia / Random Forest: accuracy_cv_media = 0.9802
  - referencia / XGBoost: accuracy_cv_media = 0.9790
  - referencia / Gradient Boosting: accuracy_cv_media = 0.9736
  - referencia / Decision Tree: accuracy_cv_media = 0.9694
  - referencia / Logistic Regression: accuracy_cv_media = 0.8993
  - comportamental / XGBoost: accuracy_cv_media = 0.8095
  - comportamental / Random Forest: accuracy_cv_media = 0.7963
  - comportamental / Gradient Boosting: accuracy_cv_media = 0.7579

Combinações que NÃO atingem a meta (pela validação cruzada):
  - comportamental / Decision Tree: accuracy_cv_media = 0.7268
  - comportamental / Logistic Regression: accuracy_cv_media = 0.6075


## 8. Melhor modelo por cenário

Selecionamos o melhor modelo de cada cenário a partir do `ranking` já ordenado por `accuracy_cv_media`/`f1_macro_cv_media` (Seção 7) — ou seja, pela validação cruzada, nunca pelo teste. Em seguida, para cada melhor modelo, recuperamos o `classification_report` completo diretamente do dicionário `detalhes` (calculado na Seção 6, com a ordem clínica de `ORDEM_NIVEIS_OBESIDADE`) — sem re-treinar nenhuma pipeline.

In [10]:
melhor_por_cenario = ranking.groupby("cenario", as_index=False).first()
melhor_por_cenario

,cenario,modelo,accuracy_cv_media,accuracy_cv_desvio,f1_macro_cv_media,f1_macro_cv_desvio,accuracy_teste,f1_macro_teste,fit_time,bate_meta_cv,bate_meta_teste
0,comportamental,XGBoost,0.809466,0.009409,0.803079,0.009721,0.775120,0.769354,3.099699,True,True
1,referencia,Random Forest,0.980232,0.006441,0.979263,0.006703,0.976077,0.975247,0.647997,True,True


In [11]:
for _, linha in melhor_por_cenario.iterrows():
    nome_cenario = linha["cenario"]
    nome_modelo = linha["modelo"]

    avaliacao_teste = detalhes[(nome_cenario, nome_modelo)]["avaliacao_teste"]

    print(f"{'=' * 60}")
    print(f"Melhor modelo do cenário '{nome_cenario}': {nome_modelo}")
    print(f"{'=' * 60}")
    print(
        f"Validação cruzada -> accuracy: {linha['accuracy_cv_media']:.4f} "
        f"± {linha['accuracy_cv_desvio']:.4f} | "
        f"F1-macro: {linha['f1_macro_cv_media']:.4f} "
        f"± {linha['f1_macro_cv_desvio']:.4f}"
    )
    print(
        f"Teste (informativo)-> accuracy: {linha['accuracy_teste']:.4f} | "
        f"F1-macro: {linha['f1_macro_teste']:.4f}"
    )
    print("\nClassification report (teste):")
    print(avaliacao_teste["classification_report"])
    print()

Melhor modelo do cenário 'comportamental': XGBoost
Validação cruzada -> accuracy: 0.8095 ± 0.0094 | F1-macro: 0.8031 ± 0.0097
Teste (informativo)-> accuracy: 0.7751 | F1-macro: 0.7694

Classification report (teste):
                     precision    recall  f1-score   support

Insufficient_Weight       0.84      0.87      0.85        53
      Normal_Weight       0.59      0.58      0.58        57
 Overweight_Level_I       0.67      0.69      0.68        55
Overweight_Level_II       0.78      0.60      0.68        58
     Obesity_Type_I       0.73      0.76      0.74        70
    Obesity_Type_II       0.83      0.92      0.87        60
   Obesity_Type_III       0.97      0.98      0.98        65

           accuracy                           0.78       418
          macro avg       0.77      0.77      0.77       418
       weighted avg       0.77      0.78      0.77       418


Melhor modelo do cenário 'referencia': Random Forest
Validação cruzada -> accuracy: 0.9802 ± 0.0064 | F1-macr

## 9. Conclusões

**Cenário `referencia`:** todos os algoritmos ficam muito acima da meta de 75%, como esperado (teto de performance dado o vazamento de IMC). O melhor pela validação cruzada é o **Random Forest** (accuracy_cv_media ≈ 98,0% ± 0,6 p.p.; F1-macro_cv_media ≈ 97,9%), seguido de perto pelo **XGBoost** (accuracy_cv_media ≈ 97,9% ± 0,4 p.p.). No teste, a ordem até se inverte (XGBoost ≈ 98,8% vs. Random Forest ≈ 97,6%), mas isso não muda a escolha do melhor modelo, que é feita pela validação cruzada — outra ilustração prática de por que o teste não deve ser usado como critério de seleção. Continua valendo a ressalva do `03_modelo_baseline.ipynb`: este cenário não é candidato a produção, por reaprender a fórmula do IMC em vez de agregar valor clínico.

**Cenário `comportamental` (o que importa para a meta do desafio):** o **XGBoost** é o melhor pela validação cruzada, com accuracy_cv_media ≈ **80,9% ± 0,9 p.p.** e F1-macro_cv_media ≈ 80,3% — **acima da meta de 75%**, e confirmado no teste (accuracy ≈ 77,5%, F1-macro ≈ 76,9%, também acima de 75%). O **Random Forest** fica logo atrás na validação cruzada (accuracy_cv_media ≈ 79,6%), mas com teste ligeiramente maior que o do XGBoost (≈ 79,4%). O **Gradient Boosting** também ultrapassa a meta por uma margem pequena (accuracy_cv_media ≈ 75,8%). A **Árvore de Decisão** fica abaixo (≈ 72,7%), e a **Regressão Logística** (baseline) confirma o resultado da fase 03, bem abaixo (≈ 60,8% CV / 62,2% teste).

**A meta de 75% de accuracy foi atingida no cenário comportamental.** Diferente do baseline (Regressão Logística, ~62,2% no teste), três dos quatro algoritmos mais robustos testados aqui (XGBoost, Random Forest e Gradient Boosting) superam os 75% exigidos usando apenas hábitos de vida, sem qualquer métrica antropométrica — o ganho vem de algoritmos capazes de capturar relações não lineares entre as variáveis de hábito, que a Regressão Logística (linear) não conseguia representar.

**Overfitting (gap CV vs. teste):** no cenário `referencia`, o gap é pequeno e até favorável ao teste (Random Forest: 98,0% CV vs. 97,6% teste; XGBoost: 97,9% CV vs. 98,8% teste) — sem sinal de overfitting. No cenário `comportamental`, o **XGBoost** (melhor pela CV) tem um gap mais chamativo: 80,9% CV vs. 77,5% teste, uma queda de ~3,4 p.p. — bem maior que o desvio-padrão entre folds (0,9 p.p.), sugerindo alguma variância do split de teste ou leve tendência a overfitting do XGBoost com hiperparâmetros padrão (sem tuning). Já o **Random Forest** tem um gap quase nulo (79,6% CV vs. 79,4% teste), indicando uma generalização mais estável.

**Tempo de treinamento como desempate:** o XGBoost é o modelo mais lento entre os testados no cenário comportamental (~2,6s), cerca de **9 vezes mais lento** que o Random Forest (~0,28s) para uma vantagem de apenas ~1,3 p.p. de accuracy_cv_media — e, como visto acima, com um gap CV-teste maior. Combinando desempenho de validação cruzada, estabilidade (gap CV-teste) e custo de treino, o **Random Forest é um candidato pelo menos tão forte quanto o XGBoost** para produção no cenário comportamental, mesmo o XGBoost sendo o "melhor" pelo critério estrito de `accuracy_cv_media` usado aqui.

**Recall de `Overweight_Level_II` (a classe mais difícil no baseline):** no baseline da fase 03 (Regressão Logística, cenário comportamental), o recall dessa classe era de apenas **0,16** — o modelo praticamente não conseguia identificar esses pacientes. No melhor modelo deste notebook (XGBoost, cenário comportamental), o recall da mesma classe sobe para **0,60** — quase 4x maior. É a evidência mais concreta de que os algoritmos mais robustos não apenas elevam a accuracy geral, mas corrigem especificamente a maior fraqueza do baseline: distinguir os níveis intermediários de sobrepeso/obesidade a partir de hábitos de vida.

**Recomendação final:** para o cenário comportamental (o clinicamente relevante para o hospital), o baseline oficial do `03_modelo_baseline.ipynb` está superado — a meta de 75% foi atingida. Como próximo passo, vale considerar o Random Forest como alternativa ao XGBoost (desempenho de teste equivalente ou superior, gap CV-teste menor, treino ~9x mais rápido) antes de decidir o modelo final a seguir para a etapa de deploy no Streamlit.